In [1]:
# %pip install fsspec s3fs openpyxl

In [2]:
# %pip install boto3

In [3]:
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql.functions import col, count, when, to_timestamp
from pyspark.sql import functions as F


spark = SparkSession.builder \
    .master("local[*]") \
    .appName("S3") \
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4"
    ) \
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    ) \
    .config(
        "spark.hadoop.fs.s3a.aws.profile",
        "default"
    ) \
    .config("spark.driver.memory", "10g") \
    .config("spark.driver.memoryOverhead", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

In [4]:
import boto3

session = boto3.Session(profile_name="default")

print(session.client("sts").get_caller_identity())

{'UserId': 'AIDA3MO6D2ITBKTYS7MOL', 'Account': '782686736934', 'Arn': 'arn:aws:iam::782686736934:user/alan', 'ResponseMetadata': {'RequestId': 'eabaa1cb-797c-4032-9e67-ac931a6037f4', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'eabaa1cb-797c-4032-9e67-ac931a6037f4', 'x-amz-sts-extended-request-id': 'MTp1cy1lYXN0LTI6UzoxNzg4ODQ4NTc4NjMwOlI6OHN0bjREY1o=', 'content-type': 'text/xml', 'content-length': '401', 'date': 'Tue, 08 Sep 2026 06:22:58 GMT'}, 'RetryAttempts': 0}}


In [5]:
spark.sparkContext._jsc.hadoopConfiguration().set(
    "fs.s3a.aws.credentials.provider",
    "com.amazonaws.auth.profile.ProfileCredentialsProvider"
)

spark.sparkContext._jsc.hadoopConfiguration().set(
    "fs.s3a.aws.profile",
    "default"
)

print(
    spark.sparkContext._jsc.hadoopConfiguration().get(
        "fs.s3a.aws.credentials.provider"
    )
)

print(
    spark.sparkContext._jsc.hadoopConfiguration().get(
        "fs.s3a.aws.profile"
    )
)

com.amazonaws.auth.profile.ProfileCredentialsProvider
default


In [6]:
df_accounts = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("s3a://datapath-buckets/bank_datasets/raw/HI-Small_accounts.csv")

df_accounts.show(5)

+--------------------+-------+--------------+---------+--------------------+
|           Bank Name|Bank ID|Account Number|Entity ID|         Entity Name|
+--------------------+-------+--------------+---------+--------------------+
| Portugal Bank #4507| 331579|     80B779D80|80062E240|Sole Proprietorsh...|
|     Canada Bank #27|    210|     809D86900|800C998A0|  Corporation #33520|
|         UK Bank #33|  21884|     80812BE00|800C47F50|  Partnership #35397|
|  Germany Bank #4815|  32742|     81047F300|80096F0B0|  Corporation #48813|
|National Bank of ...| 127390|     80BD8CF00|800FB8760|    Corporation #889|
+--------------------+-------+--------------+---------+--------------------+
only showing top 5 rows



In [7]:
df_transactions = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("s3a://datapath-buckets/bank_datasets/raw/HI-Small_Trans.csv")

df_transactions.show(5)

+----------------+---------+---------+-------+---------+---------------+------------------+-----------+----------------+--------------+-------------+
|       Timestamp|From Bank| Account2|To Bank| Account4|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|
+----------------+---------+---------+-------+---------+---------------+------------------+-----------+----------------+--------------+-------------+
|2022/09/01 00:20|       10|8000EBD30|     10|8000EBD30|        3697.34|         US Dollar|    3697.34|       US Dollar|  Reinvestment|            0|
|2022/09/01 00:20|     3208|8000F4580|      1|8000F5340|           0.01|         US Dollar|       0.01|       US Dollar|        Cheque|            0|
|2022/09/01 00:00|     3209|8000F4670|   3209|8000F4670|       14675.57|         US Dollar|   14675.57|       US Dollar|  Reinvestment|            0|
|2022/09/01 00:02|       12|8000F5030|     12|8000F5030|        2806.97|         US Dollar|    2806.

In [8]:
# Revisa el schema inferido
df_accounts.printSchema()
df_transactions.printSchema()

root
 |-- Bank Name: string (nullable = true)
 |-- Bank ID: integer (nullable = true)
 |-- Account Number: string (nullable = true)
 |-- Entity ID: string (nullable = true)
 |-- Entity Name: string (nullable = true)

root
 |-- Timestamp: string (nullable = true)
 |-- From Bank: integer (nullable = true)
 |-- Account2: string (nullable = true)
 |-- To Bank: integer (nullable = true)
 |-- Account4: string (nullable = true)
 |-- Amount Received: double (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- Is Laundering: integer (nullable = true)



In [9]:
print("Número de registros en df_accounts:", df_accounts.count())
print("Número de registros en df_transactions:", df_transactions.count())


Número de registros en df_accounts: 518581
Número de registros en df_transactions: 5078345


In [10]:
# Auditoría de nulos y duplicados
df_accounts.select([count(when(col(c).isNull(), c)).alias(c) for c in df_accounts.columns]).show()

df_transactions.select([count(when(col(c).isNull(), c)).alias(c) for c in df_transactions.columns]).show()

+---------+-------+--------------+---------+-----------+
|Bank Name|Bank ID|Account Number|Entity ID|Entity Name|
+---------+-------+--------------+---------+-----------+
|        0|      0|             0|        0|          0|
+---------+-------+--------------+---------+-----------+

+---------+---------+--------+-------+--------+---------------+------------------+-----------+----------------+--------------+-------------+
|Timestamp|From Bank|Account2|To Bank|Account4|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|
+---------+---------+--------+-------+--------+---------------+------------------+-----------+----------------+--------------+-------------+
|        0|        0|       0|      0|       0|              0|                 0|          0|               0|             0|            0|
+---------+---------+--------+-------+--------+---------------+------------------+-----------+----------------+--------------+-------------+



In [11]:
def duplicados(df, nombre):
    total_filas = df.count()
    filas_unicas = df.distinct().count()
    duplicados = total_filas - filas_unicas

    if total_filas != filas_unicas:
        print(f"El DataFrame {nombre} tiene duplicados")
        print(f"Filas duplicadas: {duplicados}")
        print(f"Porcentaje de duplicadas: {duplicados / total_filas * 100:.2f}%")
    else:
        print(f"{nombre}: No hay duplicados")

duplicados(df_accounts, "df_accounts")
duplicados(df_transactions, "df_transactions")

df_accounts: No hay duplicados
El DataFrame df_transactions tiene duplicados
Filas duplicadas: 9
Porcentaje de duplicadas: 0.00%


In [12]:
df_transactions = df_transactions.dropDuplicates()


In [13]:


df_transactions = df_transactions.withColumn(
    "Timestamp",
    to_timestamp(col("Timestamp"), "yyyy/MM/dd HH:mm")
)



In [14]:

df_transactions = df_transactions.withColumn("minute_part", F.minute("Timestamp"))
df_transactions = df_transactions.withColumn("day_of_month", F.dayofmonth("Timestamp"))
df_transactions = df_transactions.withColumn("month", F.month("Timestamp"))
df_transactions = df_transactions.withColumn("year", F.year("Timestamp"))


In [15]:
df_transactions=df_transactions.withColumnRenamed("Account2", "Account_From")
df_transactions=df_transactions.withColumnRenamed("Account4", "Account_To")

In [16]:
df_transactions.printSchema()

root
 |-- Timestamp: timestamp (nullable = true)
 |-- From Bank: integer (nullable = true)
 |-- Account_From: string (nullable = true)
 |-- To Bank: integer (nullable = true)
 |-- Account_To: string (nullable = true)
 |-- Amount Received: double (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- Is Laundering: integer (nullable = true)
 |-- minute_part: integer (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)



In [17]:
df_transactions.select("Payment Currency").distinct().show()

df_transactions.select("Receiving Currency").distinct().show()

+-----------------+
| Payment Currency|
+-----------------+
|        US Dollar|
|             Euro|
|             Yuan|
|              Yen|
|         UK Pound|
|      Brazil Real|
|  Canadian Dollar|
|          Bitcoin|
|Australian Dollar|
|            Rupee|
|      Saudi Riyal|
|     Mexican Peso|
|      Swiss Franc|
|            Ruble|
|           Shekel|
+-----------------+

+------------------+
|Receiving Currency|
+------------------+
|         US Dollar|
|              Euro|
|              Yuan|
|               Yen|
|          UK Pound|
|       Brazil Real|
|   Canadian Dollar|
|           Bitcoin|
| Australian Dollar|
|             Rupee|
|       Saudi Riyal|
|      Mexican Peso|
|       Swiss Franc|
|             Ruble|
|            Shekel|
+------------------+



In [18]:

# 1.42% de las transacciones son cross-currency

df_transactions.filter(
    col("Payment Currency") != col("Receiving Currency")
).count()

72166

In [19]:
df_transactions.groupBy('Payment Currency').count().show()   

+-----------------+-------+
| Payment Currency|  count|
+-----------------+-------+
|        US Dollar|1895169|
|             Yuan| 213752|
|              Yen| 155209|
|             Euro|1168296|
|         UK Pound| 180738|
|      Brazil Real|  70703|
|  Canadian Dollar| 140042|
|          Bitcoin| 146061|
|            Rupee| 190202|
|Australian Dollar| 136769|
|      Saudi Riyal|  89014|
|     Mexican Peso| 110159|
|      Swiss Franc| 234860|
|            Ruble| 155178|
|           Shekel| 192184|
+-----------------+-------+



In [23]:
df_transactions.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet("s3a://datapath-buckets/bank_datasets/curated/transactions/")

In [ ]:

df_accounts.write \
    .mode("overwrite") \
    .parquet("s3a://datapath-buckets/bank_datasets/curated/accounts/") 
